## 16. EAD + Expected Loss

In [ ]:
# 1. VECTORIZED EAD — amortization formula
# EAD = remaining balance = loan_amnt × [(1+r)^n - (1+r)^t] / [(1+r)^n - 1]
print("Computing EAD...")
if 'term' in df.columns:
    all_term = df.loc[df_cleaned.index, 'term'].str.extract(r'(\d+)').astype(float).squeeze()
else:
    all_term = pd.Series(36, index=df_cleaned.index)

all_duration = survival_df['duration_months'].reindex(df_cleaned.index).fillna(18)
all_int_rate = df_cleaned['int_rate']
all_loan_amnt = df.loc[df_cleaned.index, 'loan_amnt']

mr = all_int_rate / 12 / 100
n = all_term
t = all_duration.clip(upper=all_term)
num = (1+mr)**n - (1+mr)**t
den = (1+mr)**n - 1
ead_fraction = np.where(den==0, np.maximum(1-t/n,0), (num/den).clip(0,1))
all_ead_fraction = pd.Series(ead_fraction, index=df_cleaned.index)
all_ead = all_loan_amnt * all_ead_fraction

# 2. PD — calibrated LightGBM
print("Computing PD...")
all_pd = pd.Series(calibrated_lgb.predict_proba(df_cleaned[feature_cols])[:,1], index=df_cleaned.index)

# 3. LGD — two-stage model with leakage-free inputs
print("Computing LGD...")
lgd_input = df_cleaned[[c for c in lgd_features_v2 if c in df_cleaned.columns]].copy()
lgd_input['loan_age_at_default'] = all_duration
lgd_input['payment_ratio'] = (1 - all_ead_fraction).clip(0, 1)  # amortization-implied, no leakage
lgd_input['recoveries_ratio'] = 0  # 0 for performing loans
lgd_input['months_paid_ratio'] = all_duration.clip(lower=0)
lgd_input = lgd_input[lgd_features_v2].fillna(lgd_input.median())
all_lgd = pd.Series(predict_lgd_v2(lgd_input), index=df_cleaned.index)

# Blend: PD-weighted average of model LGD and empirical mean (corrects distribution shift)
EMPIRICAL_MEAN_LGD = defaulted_df['lgd'].mean()
all_lgd = all_pd * all_lgd + (1 - all_pd) * EMPIRICAL_MEAN_LGD
print(f"Mean LGD after blend: {all_lgd.mean():.4f}")

# 4. EXPECTED LOSS = PD × LGD × EAD
el_df = pd.DataFrame({
    'loan_amnt': all_loan_amnt, 'pd': all_pd, 'lgd': all_lgd,
    'ead': all_ead, 'ead_fraction': all_ead_fraction,
    'grade': df.loc[df_cleaned.index, 'grade']
}, index=df_cleaned.index)
el_df['expected_loss'] = el_df['pd'] * el_df['lgd'] * el_df['ead']
el_df['el_rate'] = el_df['expected_loss'] / el_df['loan_amnt']

print(f"\n--- Portfolio Expected Loss Summary ---")
print(f"Total Exposure: ${el_df['loan_amnt'].sum():,.0f}")
print(f"Total EL:       ${el_df['expected_loss'].sum():,.0f}")
print(f"EL Rate:        {el_df['expected_loss'].sum()/el_df['loan_amnt'].sum():.2%}")

grade_summary = el_df.groupby('grade').agg(
    mean_pd=('pd','mean'), mean_lgd=('lgd','mean'),
    mean_ead_frac=('ead_fraction','mean'), el_rate=('el_rate','mean'),
    total_el=('expected_loss','sum')
).round(4)
print(f"\n--- EL by Grade ---")
print(grade_summary)

grade_summary['el_rate'].plot(kind='bar', color='darkred', alpha=0.7, title='Expected Loss Rate by Grade')
plt.ylabel('EL Rate')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# EL by Vintage
el_df['issue_year'] = pd.to_datetime(df.loc[df_cleaned.index,'issue_d'], format='mixed').dt.year
print("--- EL by Vintage ---")
print(el_df.groupby('issue_year').agg(
    mean_pd=('pd','mean'), mean_lgd=('lgd','mean'),
    el_rate=('el_rate','mean'), total_el=('expected_loss','sum'), count=('loan_amnt','count')
).round(4))

# EL by Purpose
el_df['purpose'] = df['purpose'].reindex(df_cleaned.index)
print("\n--- EL by Purpose ---")
print(el_df.groupby('purpose').agg(
    el_rate=('el_rate','mean'), total_el=('expected_loss','sum'), count=('loan_amnt','count')
).round(4).sort_values('el_rate', ascending=False))

# EL Distribution + Tail
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(el_df['el_rate'], bins=100, color='#e74c3c', alpha=0.8)
axes[0].axvline(el_df['el_rate'].mean(), color='black', linestyle='--',
                label=f"Mean: {el_df['el_rate'].mean():.4f}")
axes[0].axvline(el_df['el_rate'].quantile(0.95), color='gold', linestyle='--',
                label=f"95th: {el_df['el_rate'].quantile(0.95):.4f}")
axes[0].set_title('EL Rate Distribution — Full Portfolio')
axes[0].legend()

tail = el_df[el_df['el_rate'] >= el_df['el_rate'].quantile(0.95)]
axes[1].hist(tail['el_rate'], bins=50, color='#c0392b', alpha=0.8)
axes[1].set_title(f'EL Tail (Top 5%) — {len(tail):,} loans, ${tail["expected_loss"].sum():,.0f} total EL')
plt.tight_layout()
plt.show()

print(f"95th pct EL rate: {el_df['el_rate'].quantile(0.95):.4f}")
print(f"Top 5% loans: {tail['expected_loss'].sum()/el_df['expected_loss'].sum():.1%} of total EL")

### Reading the Expected Loss Number

**Portfolio EL = $1.34B on $19.4B exposure (6.89% EL rate)**

Three modeling choices drive this number:

- **PD** — Calibrated LightGBM. Range: 6% (Grade A) to 49% (Grade G).
- **LGD** — Two-stage model blended with empirical mean (0.41) weighted by PD. Blend corrects for distribution shift between defaulter training population and performing loan prediction population.
- **EAD** — Amortization formula. `payment_ratio` uses scheduled paydown (1 − EAD fraction) rather than observed `total_pymnt` to avoid leakage on non-defaulted loans.

Grade G EL rate (24.65%) is **22x Grade A (1.09%)** — driven by simultaneously higher PD, LGD, and EAD. The multiplicative effect of all three components explains the dramatic grade spread. This EL figure represents the minimum loan loss reserve under IFRS 9 Stage 1 provisioning.